# Pre-body

## Clearing past runs (optional)

In [1]:
!rm -rf logs/ # clear logs
!rm -rf optimizer_output/

## IIC-OSIC Env Setup

In [17]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [18]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

05:04:58 - SymXplorer.jupyter: [INFO] Spicelib_Wrapper imported successfully.


# Instantiations


## Loading the project config

In [19]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

05:04:59 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
05:04:59 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/logs/SymXplorer_2025-10-06_05-04-59.log
05:04:59 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
05:04:59 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/project_setup.yaml
05:04:59 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: TwoPointsDE, type=nevergrad, budget=100, random_seed=48
05:04:59 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
05:04:59 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
05:04:59 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
05:04:59 - SymXplorer.domains: [INFO] 	Number of target specs: 3
05:04:59 - SymXplorer.domains: [INFO] 		- TargetSpec(name=ugf, target=200e6, range=1.00e+08 toleran

Project_Setup(name='5T-OTA', description='5 Transistor OTA example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2'), netlist=PosixPath('spice/ota-5t_tb-loopgain.spice'), outdir=PosixPath('sizing/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07)}), pvt=PVT(temp=25, corner='tt', supply=1.8), dut_params=[Param(name='x_dut_nfet_input_w', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False), Param(name='x_dut_nfet_input_l', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06)

## Create a SPICE simulator wrapper

In [20]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

05:04:59 - SymXplorer.spicelib: [INFO] 📂 Creating output directory for the first time: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/optimizer_output
05:04:59 - SymXplorer.spicelib: [INFO] --------------------------------------------------
05:04:59 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
05:04:59 - SymXplorer.spicelib: [INFO] 	📝 Project: 5T-OTA
05:04:59 - SymXplorer.spicelib: [INFO] 	📜 Schematic: ota-5t_tb-loopgain
05:04:59 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/optimizer_output
05:04:59 - SymXplorer.spicelib: [INFO] --------------------------------------------------
05:04:59 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
05:04:59 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
05:04:59 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['v_dd', 'GND', 'v_ss', 'v_in', 'v_ena', 'vr1', 'net1', 'vf1', 'net2', 'net3', 'net4

## Create an optimizer object

In [21]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

05:04:59 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs


## Sanity Check

In [22]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

05:04:59 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
05:04:59 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
05:05:00 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/optimizer_output/sanity_check/5T-OTA_sanity.log
05:05:00 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/optimizer_output/sanity_check/5T-OTA_sanity.raw
05:05:00 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
05:05:00 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Main Body

## Optimization

In [23]:
circuit_optimizer.parameterize()

Dict(x_dut_nfet_input_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_input_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_mirror_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_mirror_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_pfet_load_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_pfet_load_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_input_w': 50.0, 'x_dut_nfet_input_l': 50.0, 'x_dut_nfet_mirror_w': 50.0, 'x_dut_nfet_mirror_l': 50.0, 'x_dut_pfet_load_w': 50.0, 'x_dut_pfet_load_l': 50.0}

In [24]:
circuit_optimizer.optimize()

05:05:00 - SymXplorer.optimizer: [INFO] Optimization process started.
05:05:00 - SymXplorer.optimizer: [INFO] Optimizer is set to TwoPointsDE with budget = 100
Optimizing: 100%|██████████| 100/100 [00:47<00:00,  2.10trial/s]
05:05:47 - SymXplorer.optimizer: [INFO] Optimization process completed.


[{'params': {'x_dut_nfet_input_w': 13.008601068070769,
   'x_dut_nfet_input_l': 1.0588247620941238,
   'x_dut_nfet_mirror_w': 98.29140062557896,
   'x_dut_nfet_mirror_l': 36.25990667079719,
   'x_dut_pfet_load_w': 73.89490333304465,
   'x_dut_pfet_load_l': 4.449617001059824},
  'loss': np.float64(99.28833292530426),
  'metadata': {'ugf': {'curr_val': np.float64(617557200.0),
    'loss': np.float64(96.66021825462201)},
   'dcgain': {'curr_val': np.float64(24.74256),
    'loss': np.float64(2.6281146706822467)},
   'pm': {'curr_val': np.float64(95.0), 'loss': np.float64(0.0)}}},
 {'params': {'x_dut_nfet_input_w': 25.266105610430177,
   'x_dut_nfet_input_l': 66.79100889381697,
   'x_dut_nfet_mirror_w': 67.73732158556467,
   'x_dut_nfet_mirror_l': 44.204249159864695,
   'x_dut_pfet_load_w': 4.7275418581637645,
   'x_dut_pfet_load_l': 90.61950213562862},
  'loss': np.float64(71.22035544014598),
  'metadata': {'ugf': {'curr_val': np.float64(24592320.0),
    'loss': np.float64(67.8882561262637

In [25]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

05:05:48 - SymXplorer.plotter: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/loss_curve.html
05:05:48 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [26]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
# metadata

05:05:48 - SymXplorer.optimizer: [INFO] best loss: 1.0253340662698784


In [27]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

x_dut_nfet_input_w: 4.50e-06
x_dut_nfet_input_l: 1.47e-06
x_dut_nfet_mirror_w: 6.50e-06
x_dut_nfet_mirror_l: 5.62e-06
x_dut_pfet_load_w: 3.38e-06
x_dut_pfet_load_l: 3.05e-06


In [28]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="out")

05:05:48 - SymXplorer.optimizer: [INFO] total loss: 1.0253340662698784
05:05:48 - SymXplorer.optimizer: [INFO] 	Spec 'ugf': curr_val=206970000.0, loss=0.0
05:05:48 - SymXplorer.optimizer: [INFO] 	Spec 'dcgain': curr_val=27.94926, loss=1.0253340662698784
05:05:48 - SymXplorer.optimizer: [INFO] 	Spec 'pm': curr_val=100.0, loss=0.0


### (3) Metric Trace

In [29]:
_ = circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='ugf', show=True)

05:05:49 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


In [30]:
circuit_optimizer.plot_loss_value_by_spec(spec_name="dcgain", show=True)
circuit_optimizer.plot_loss_value_by_spec(spec_name="ugf", show=True)
circuit_optimizer.plot_loss_value_by_spec(spec_name="pm", show=True)

05:05:49 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 4.499429960616141
05:05:49 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


05:05:49 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 99.34606392106429
05:05:49 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


05:05:49 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 27.08471185167214
05:05:49 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


### (4) Design Space Exploration

In [ ]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_pfet_load_w", param_y="x_dut_pfet_load_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_input_w", param_y="x_dut_nfet_input_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_mirror_w", param_y="x_dut_nfet_mirror_l", show=True)

05:07:26 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


05:07:28 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...
